# GoMeal ML Brain Learning

This notebook documents the learning side of GoMeal ML:

- `Neuron`: user or recipe/post memory unit
- `Pattern`: repeated coactivation key
- `MacroNeuron`: stable abstraction created after repeated patterns
- `recipe_coactivation_counts`: multi-user collaborative memory

The production files this maps to are:

- `brain/core/network.py`
- `brain/core/core/neuron.py`
- `brain/core/core/macro_neuron.py`
- `brain/core/patterns/pattern.py`
- `brain/core/patterns/pattern_manager.py`
- `brain/api/subscriber.py`

## Config Fields

Add these to `brain/core/config/network_config.yaml`:

```yaml
macro_pattern_min_count: 5
multi_user_recent_window: 10
multi_user_min_coactivation: 3
multi_user_boost: 0.12
```

And add matching Pydantic fields to `NeuralNetworkConfig` in `brain/core/config/config.py`.

In [ ]:
from dataclasses import dataclass, field
from typing import Dict, Tuple, List, Optional
import itertools


@dataclass
class Config:
    macro_pattern_min_count: int = 5
    multi_user_recent_window: int = 10
    multi_user_min_coactivation: int = 3
    multi_user_boost: float = 0.12
    learning_rate: float = 0.03


config = Config()
config

In [ ]:
_ids = itertools.count(1)


@dataclass
class Neuron:
    source_model: str
    source_id: int
    vector: List[float]
    neuron_id: int = field(default_factory=lambda: next(_ids))
    connections: List[Tuple[str, int]] = field(default_factory=list)


@dataclass
class MacroNeuron:
    child_neuron_ids: List[str]
    vector: List[float]
    strength: float = 1.0
    macro_neuron_id: int = field(default_factory=lambda: next(_ids))


def mean_vectors(vectors: List[List[float]]) -> List[float]:
    return [sum(v[i] for v in vectors) / len(vectors) for i in range(len(vectors[0]))]


def create_macro_neuron(neurons: List[Neuron]) -> MacroNeuron:
    return MacroNeuron(
        child_neuron_ids=[str(n.neuron_id) for n in neurons],
        vector=mean_vectors([n.vector for n in neurons]),
    )

In [ ]:
@dataclass
class Pattern:
    neuron_ids: List[int]
    activation_count: int = 1

    def __post_init__(self):
        self.neuron_ids = sorted(int(nid) for nid in self.neuron_ids)
        self.id = "-".join(str(nid) for nid in self.neuron_ids)

    def increment(self):
        self.activation_count += 1


class PatternManager:
    def __init__(self):
        self.patterns: Dict[str, Pattern] = {}

    def register_pattern(self, neuron_ids: List[int]) -> Pattern:
        key = "-".join(str(nid) for nid in sorted(int(nid) for nid in neuron_ids))
        if key in self.patterns:
            self.patterns[key].increment()
        else:
            self.patterns[key] = Pattern(neuron_ids)
        return self.patterns[key]

In [ ]:
class Brain:
    def __init__(self):
        self.neurons: Dict[int, Neuron] = {}
        self.neurons_source: Dict[Tuple[str, int], Neuron] = {}
        self.macro_neurons: Dict[int, MacroNeuron] = {}
        self.pattern_manager = PatternManager()
        self.user_recent_recipe_activations: Dict[int, List[int]] = {}
        self.recipe_coactivation_counts: Dict[Tuple[int, int], int] = {}

    def add_neuron(self, neuron: Neuron) -> Neuron:
        self.neurons[neuron.neuron_id] = neuron
        self.neurons_source[(neuron.source_model, neuron.source_id)] = neuron
        return neuron

    def get_neuron(self, neuron_id: int) -> Optional[Neuron]:
        return self.neurons.get(neuron_id)

    def get_neuron_by_source_id(self, source_model: str, source_id: int) -> Optional[Neuron]:
        return self.neurons_source.get((source_model, source_id))

    def connect(self, source: Neuron, target: Neuron):
        connection = (target.source_model, target.neuron_id)
        if connection not in source.connections:
            source.connections.append(connection)

    def create_macro_neuron_from_user_neuron(self, neuron_ids: List[int], min_count: int, strength: float) -> Optional[MacroNeuron]:
        pattern = self.pattern_manager.register_pattern(neuron_ids)
        if pattern.activation_count < min_count:
            return None

        child_set = set(str(nid) for nid in pattern.neuron_ids)
        for macro in self.macro_neurons.values():
            if set(macro.child_neuron_ids) == child_set:
                macro.strength += 0.1
                return macro

        neurons = [self.get_neuron(nid) for nid in pattern.neuron_ids]
        neurons = [n for n in neurons if n is not None]
        if len(neurons) < 2:
            return None

        macro = create_macro_neuron(neurons)
        macro.strength = strength
        self.macro_neurons[macro.macro_neuron_id] = macro
        print(f"[personal_macro_created] pattern={pattern.id} macro_id={macro.macro_neuron_id}")
        return macro

    def create_macro_neuron_from_recipe_neuron(self, user_neuron: Neuron, recipe_neuron: Neuron, max_recent: int, min_count: int, strength: float) -> Optional[MacroNeuron]:
        recent = self.user_recent_recipe_activations.setdefault(user_neuron.neuron_id, [])
        created = None

        for recent_recipe_id in recent:
            if recent_recipe_id == recipe_neuron.neuron_id:
                continue

            pair = tuple(sorted([recent_recipe_id, recipe_neuron.neuron_id]))
            count = self.recipe_coactivation_counts.get(pair, 0) + 1
            self.recipe_coactivation_counts[pair] = count

            if count >= min_count:
                created = self.create_macro_neuron_from_user_neuron(list(pair), min_count=1, strength=strength)

        recent.append(recipe_neuron.neuron_id)
        self.user_recent_recipe_activations[user_neuron.neuron_id] = recent[-max_recent:]
        return created

## Subscriber Integration

In `brain/api/subscriber.py`, after `user_neuron` and `recipe_neuron` exist and after direct connection, keep these two calls:

```python
_user_macro_neuron = _brain.create_macro_neuron_from_user_neuron(
    neuron_ids=[user_neuron.neuron_id, recipe_neuron.neuron_id],
    min_count=config.macro_pattern_min_count,
    strength=neuron_strength,
    learning_rate=config.learning_rate,
)

_recipe_macro_neuron = _brain.create_macro_neuron_from_recipe_neuron(
    user_neuron=user_neuron,
    recipe_neuron=recipe_neuron,
    max_recent=config.multi_user_recent_window,
    min_count=config.multi_user_min_coactivation,
    strength=neuron_strength,
    learning_rate=config.learning_rate,
)
```

Guard debugger calls because both can be `None` before thresholds are reached:

```python
if _user_macro_neuron is not None:
    Debugger.macro_neuron(_user_macro_neuron, showPatterns=True)

if _recipe_macro_neuron is not None:
    Debugger.macro_neuron(_recipe_macro_neuron, showPatterns=True)
```

In [ ]:
def get_or_create_user(brain: Brain, user_sub: int) -> Neuron:
    return brain.get_neuron_by_source_id("user", user_sub) or brain.add_neuron(
        Neuron(source_model="user", source_id=user_sub, vector=[0.0, 0.0, 0.0])
    )


def get_or_create_recipe(brain: Brain, post_id: int, vector: List[float]) -> Neuron:
    return brain.get_neuron_by_source_id("Recipe", post_id) or brain.add_neuron(
        Neuron(source_model="Recipe", source_id=post_id, vector=vector)
    )


brain = Brain()
events = [(1, 101), (1, 202), (2, 101), (2, 202), (3, 101), (3, 202)]

for user_sub, post_id in events:
    user = get_or_create_user(brain, user_sub)
    recipe = get_or_create_recipe(brain, post_id, [1.0, post_id / 1000, 0.0])
    brain.connect(user, recipe)
    brain.create_macro_neuron_from_user_neuron([user.neuron_id, recipe.neuron_id], config.macro_pattern_min_count, 1.0)
    brain.create_macro_neuron_from_recipe_neuron(user, recipe, config.multi_user_recent_window, config.multi_user_min_coactivation, 1.0)

brain.recipe_coactivation_counts